# GSR re-score: today's pipeline on all 58 SoccerNet validation clips

Our first GSR score (5 clips, the pipeline of September) collapsed at the pitch-position step. Since
then the pitch keypoints (local v10), people model, team colours and player joining all changed, so
this re-runs the **current** pipeline, with the full-match settings, on every validation clip and
scores it with the same ladder as before (image -> pitch -> +role -> +team -> GS-HOTA). Two versions
are scored: the pipeline's own track ids, and players joined by the appearance model.

Per clip: download (~160 MB), run, join, convert, then the video is deleted again. Finished clips are
skipped after a disconnect, and the last cell scores whatever clips are finished, so it can also be
stopped early. Results: `MyDrive/Playbook/gsr/current/`.

The SoccerNet data is used to measure only (research licence), never to train.

**Runtime:** L4 GPU, about 4-5 hours for all 58 clips (~5 min each). **Runtime -> Run all**; do not
edit cells. If it disconnects, Run all again.

In [ ]:
# Cell 1 — GPU, Drive, inputs
import os, json, time, glob, shutil, subprocess, sys, gc
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> L4 GPU, then re-run.')
GPU = g.stdout.strip(); print('GPU:', GPU)
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
if not os.path.isdir(PBD):
    raise SystemExit('MyDrive/Playbook not found. Other Google account? Add a shortcut to the shared Playbook folder in My Drive.')
DEST = f'{PBD}/gsr/current'; os.makedirs(f'{DEST}/pred_tracker', exist_ok=True); os.makedirs(f'{DEST}/pred_joined', exist_ok=True)
MODEL = f'{PBD}/reid_appearance/appearance_model.pt'
NEEDED = [f'{PBD}/people_model/train/checkpoint_best_total.pth', f'{PBD}/ball_model/train/checkpoint_best_total.pth',
          f'{PBD}/field_model/roboflow_v10/weights.pt', MODEL]
missing = [p for p in NEEDED if not os.path.exists(p)]
if missing:
    raise SystemExit(f'Missing in Drive: {missing}')
print('inputs found')

In [ ]:
# Cell 2 — Our repo, packages, SoccerNet labels (all validation clips, ~290 MB)
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
!pip install -q "rfdetr[train,tensorrt]==1.11.0" "git+https://github.com/roboflow/sports.git@main" "pydantic-settings==2.4.0" "python-dotenv==1.0.1" "lap>=0.5.13,<0.6" "transformers>=4.47"
!pip install -q --no-deps sn-trackeval==0.4.0
shutil.copy('baseline.env', '.env')
import numpy as np, pandas as pd
SN = '/content/SoccerNetGS'
if not glob.glob(f'{SN}/valid/*/Labels-GameState.json'):
    subprocess.run([sys.executable, '-m', 'tools.gsr_fetch', 'labels', '--split', 'valid', '--out', SN], check=True)
CLIPS = sorted(os.path.basename(os.path.dirname(p)) for p in glob.glob(f'{SN}/valid/*/Labels-GameState.json'))
print(len(CLIPS), 'clips:', CLIPS[0], '...', CLIPS[-1])

In [ ]:
# Cell 3 — Each clip: download, pipeline, join players, convert (skips finished clips; ~5 min each)
import torch
from tools import gsr_adapter as ad
from tools.reid_appearance import Embedder, crops_from_video, sample_rows
from tools.join_pieces import piece_table, cluster, joined_ids
ACCEL = {'MODEL_ACCEL': 'tensorrt', 'BALL_MODEL_ACCEL': 'fp16'}
emb_model = None
times = []
for k, c in enumerate(CLIPS):
    if os.path.exists(f'{DEST}/pred_joined/{c}.json'):
        continue
    t0 = time.time()
    video = f'{SN}/valid/{c}.mp4'
    if not os.path.exists(video):
        subprocess.run([sys.executable, '-m', 'tools.gsr_fetch', 'frames', '--split', 'valid', '--out', SN, '--seqs', c],
                       check=True, capture_output=True)
    run = f'/content/runs/{c}'; shutil.rmtree(run, ignore_errors=True)
    r = subprocess.run([sys.executable, 'main.py', '--source-video', video, '--out-dir', run, '--no-video', '--enable-team'],
                       capture_output=True, text=True, env={**os.environ, **ACCEL})
    if r.returncode != 0 and ACCEL['MODEL_ACCEL'] == 'tensorrt':
        print('TensorRT failed, using fp16:', r.stderr[-500:]); ACCEL = {'MODEL_ACCEL': 'fp16', 'BALL_MODEL_ACCEL': 'fp16'}
        r = subprocess.run([sys.executable, 'main.py', '--source-video', video, '--out-dir', run, '--no-video', '--enable-team'],
                           capture_output=True, text=True, env={**os.environ, **ACCEL})
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-3000:])
        raise SystemExit(f'{c} failed: send Claude a screenshot of the lines above.')
    t = pd.read_csv(f'{run}/per_frame_tracks.csv', low_memory=False)
    # players joined by appearance (referees keep the tracker's ids)
    if emb_model is None:
        emb_model = Embedder(); emb_model.load_state_dict(torch.load(MODEL, map_location=emb_model.device))
    ppl = t[t.class_id.isin((1, 2)) & (t.raw_tracker_id >= 0)]
    crops, rows = crops_from_video(video, sample_rows(ppl, 'raw_tracker_id', per_id=12))
    lab = cluster(piece_table(t), emb_model.embed(crops), rows.raw_tracker_id.to_numpy(), thr=0.3)
    t['joined_id'] = joined_ids(t, lab)
    t['gsr_joined_id'] = np.where(t.joined_id > 0, t.joined_id, np.where(t.track_id >= 0, t.track_id + 10_000_000, -1))
    labels = json.load(open(f'{SN}/valid/{c}/Labels-GameState.json'))
    json.dump(ad.to_predictions(t, labels, id_col='track_id'), open(f'{DEST}/pred_tracker/{c}.json', 'w'))
    json.dump(ad.to_predictions(t, labels, id_col='gsr_joined_id'), open(f'{DEST}/pred_joined/{c}.json', 'w'))
    kpi = json.load(open(f'{run}/kpi_summary.json'))
    json.dump({'fps': kpi.get('effective_fps'), 'homography_ok_rate': kpi.get('homography_ok_rate'),
               'pieces': int(ppl.raw_tracker_id.nunique()), 'players_joined': int(t.joined_id[t.joined_id > 0].nunique()),
               'accel': ACCEL}, open(f'{DEST}/pred_joined/{c}.run.json', 'w'))
    os.remove(video); shutil.rmtree(run, ignore_errors=True); del t, ppl, crops; gc.collect()
    times.append(time.time() - t0)
    left = sum(not os.path.exists(f'{DEST}/pred_joined/{x}.json') for x in CLIPS)
    print(f'{c}: {times[-1] / 60:.1f} min, pitch {kpi.get("homography_ok_rate", 0):.0%}  |  {left} left, '
          f'about {left * np.mean(times) / 3600:.1f} h', flush=True)
print('all clips done')

In [ ]:
# Cell 4 — Score the finished clips (both versions)
from tools import gsr_eval as ge
done = [c for c in CLIPS if os.path.exists(f'{DEST}/pred_joined/{c}.json')]
systems = {'tracker_ids': f'{DEST}/pred_tracker', 'joined_players': f'{DEST}/pred_joined'}
combined = ge.evaluate(f'{SN}/valid', systems, seqs=done)
per_clip = {c: ge.evaluate(f'{SN}/valid', systems, seqs=[c]) for c in done}
json.dump({'clips': done, 'combined': combined, 'per_clip': per_clip, 'gpu': GPU},
          open(f'{DEST}/ladder.json', 'w'), indent=1)
print(f'{len(done)} clips\n' + ge.format_table(combined))
print(f'\nDone. Tell Claude the results are in {DEST}.')